In [ ]:
# https://huggingface.co/fla-hub/rwkv7-2.9B-g1
# GPU T4 x2

In [1]:
import os, torch
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")

'1'

In [2]:
!pip install -q git+https://github.com/fla-org/flash-linear-attention
!pip install -q 'transformers>=4.48.0'

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
model_path = 'fla-hub/rwkv7-2.9B-g1'
tokenizer_path = 'fla-hub/rwkv7-2.9B-g1'

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True) 
# model = model.cuda() # Supported on Nvidia/AMD/Intel eg. model.xpu()

2025-10-19 15:51:34.402517: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760889094.425400    1023 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760889094.432333    1023 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

In [6]:
# Prompt geneated by ChatGPT
# Code block tags used by SmolAgents (keep consistent with YAML)
CODE_OPEN = "{{code_block_opening_tag}}"
CODE_CLOSE = "{{code_block_closing_tag}}"

# Load the SmolAgents "code agent" system prompt (adapted from file)
SMOL_CODE_AGENT_PROMPT = r"""You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
You must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step:
- In 'Thought:' explain your reasoning and what tools you want to use.
- In 'Code:' write Python code between the special tags:
  {code_open} <python code here> {code_close}
- Use print(...) inside code to expose intermediate outputs. Those printed outputs become the 'Observation:' for the next step.
- End when you call final_answer(...) inside a code block with the result.
Rules: Always include 'Thought:' and a code block with the correct tags.
""".strip().replace("{code_open}", CODE_OPEN).replace("{code_close}", CODE_CLOSE)

In [7]:
user_prompt = "I have a glass with a no bottom and a sealed top. How can I drink from it?"

messages = [
    {"role": "user", "content": user_prompt}
]

messages_wsystem = [
    {"role": "system", "content": SMOL_CODE_AGENT_PROMPT},
    {"role": "user", "content": user_prompt}
]

In [8]:
def test(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True  # Default is True, set to False to disable thinking
    )

    print("Model input\n", text)
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=1.0,
        top_p=0.3,
        repetition_penalty=1.5
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]

In [9]:
print(test(messages))

Model input
 <|rwkv_tokenizer_end_of_text|>User: I have a glass with a no bottom and a sealed top. How can I drink from it?

Assistant: <think


/usr/local/lib/python3.11/dist-packages/fla/ops/rwkv7/fused_recurrent.py:303: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (29) < num_heads (40). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/fla/ops/rwkv7/fused_recurrent.py:303: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (29) < num_heads (40). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/fla/ops/rwkv7/fused_recurrent.py:303: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (29) < num_heads (40). This may indicate the inputs were passed in

>Okay, so I have this glass that's missing its bottom, which means there's no base to hold it up. The top is sealed, so I can't pour anything in or out through the top. Hmm, how do I drink from it? Let me think about this step by step.
First, I know that normally, when you want to drink from a glass, you put it on a flat surface like a table or countertop. But since the bottom is missing, I can't just set it down. Maybe I can use something else to support it. Like, maybe I can place it on a plate or another container that has a flat bottom. That way, the glass doesn't tip over.
Wait, but if I put it on a plate, does that help? Or maybe I can find a way to balance it without any support. I remember seeing people drink from glasses without bases before, maybe using some kind of trick. Oh, right! They might use their hands to stabilize it. So, I could cup my hand around the glass while drinking, making sure not to spill anything.
Another thought: maybe I can tilt the glass slightly so tha

In [10]:
print(test(messages_wsystem))

Model input
 <|rwkv_tokenizer_end_of_text|>System: You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
You must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step:
- In 'Thought:' explain your reasoning and what tools you want to use.
- In 'Code:' write Python code between the special tags:
  {{code_block_opening_tag}} <python code here> {{code_block_closing_tag}}
- Use print(...) inside code to expose intermediate outputs. Those printed outputs become the 'Observation:' for the next step.
- End when you call final_answer(...) inside a code block with the result.
Rules: Always include 'Thought:' and a code block with the correct tags.

User: I have a glass with a no bottom and a sealed top. How can I drink from it?

Assistant: <think


/usr/local/lib/python3.11/dist-packages/fla/ops/rwkv7/fused_recurrent.py:303: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (1) < num_heads (40). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
  warnings.warn(
This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


>Okay, let's see. The user has a glass with no bottom and a sealed top. They want to drink from it. Hmm, how do I approach this?
First, I need to figure out what kind of glass this is. A glass without a bottom usually means it's open at the bottom, maybe like a bowl or a cup that's been inverted. But if it's sealed at the top, maybe it's a container that's supposed to hold liquid but has a lid on top. Wait, but if it's sealed at the top, then there's no opening for pouring. So perhaps the user is referring to a vessel that's meant to be filled through the top, but now they're trying to drink from it by going through the top. That doesn't make sense unless... maybe the glass is designed to be poured into another container, but they're trying to drink directly from it. Or maybe it's a decorative piece that's not intended for drinking.
Wait, the problem says "a glass with no bottom and a sealed top." So the glass is shaped like a cup, but the bottom part is missing. So the rim is exposed.